# Credit Card Fraud Detection (OpenShift AI + MLflow)

Same flow as the [AI on OpenShift demo](https://ai-on-openshift.io/demos/credit-card-fraud-detection-mlflow/credit-card-fraud/#3-train-the-model), with **ONNX export updated** for current workbench images (TensorFlow 2.16+ / Keras 3): SavedModel export + `tf2onnx` CLI instead of `tf2onnx.convert.from_keras` (which breaks with `keras_tensor_…` / `KeyError`).

**Data:** place `card_transdata.csv` in `../data/` (clone [credit-fraud-detection-demo](https://github.com/red-hat-data-services/credit-fraud-detection-demo) or copy the CSV from there).

In [1]:
!pip install pip -qU
!pip install -r requirements.txt -q
!pip install s3fs==2024.9.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.26.0 requires botocore<1.41.6,>=1.41.0, but you have botocore 1.40.76 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 247.2 MB/s  0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.40.76
    Uninstalling botocore-1.40.76:
      Successfully uninstalled botocore-1.40.76
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.40.76 requires botocore<1.41.0,>=1.40.76, but you have botocore 1.41.5 which is incompatible.


In [2]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from tensorflow.keras.layers import Activation, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential

import matplotlib.pyplot as plt
import mlflow
import onnx
import seaborn as sns

I0000 00:00:1786637250.399911    1743 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786637250.434937    1743 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786637251.705793    1743 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
mlflow.__version__

'3.15.1'

In [4]:
mlflow_tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
if not mlflow_tracking_uri:
    raise RuntimeError(
        "MLFLOW_TRACKING_URI is not set. Edit the workbench and connect the MLflow instance."
    )

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes-namespaced"
mlflow.set_tracking_uri(mlflow_tracking_uri)

In [5]:
def keras_model_to_onnx(model: tf.keras.Model, opset: int = 13) -> onnx.ModelProto:
    """Convert a trained Keras model to ONNX for OpenShift AI Model Serving.

    Uses a SavedModel directory + the supported ``python -m tf2onnx.convert`` CLI
    (avoids ``tf2onnx.convert.from_keras`` issues on TF 2.16+ / Keras 3).
    Note: tf2onnx expects ``--signature_def`` (underscore), not ``--signature-def``.
    """
    export_dir = tempfile.mkdtemp(prefix="cc_fraud_savedmodel_")
    onnx_fd, onnx_path = tempfile.mkstemp(suffix=".onnx")
    os.close(onnx_fd)
    try:
        if hasattr(model, "export"):
            model.export(export_dir)
        else:
            tf.saved_model.save(model, export_dir)

        loaded = tf.saved_model.load(export_dir)
        sig_keys = list(loaded.signatures.keys())
        if not sig_keys:
            raise RuntimeError("SavedModel has no signatures; cannot convert to ONNX.")
        if "serve" in sig_keys:
            signature = "serve"
        elif "serving_default" in sig_keys:
            signature = "serving_default"
        else:
            signature = sig_keys[0]

        cmd = [
            sys.executable,
            "-m",
            "tf2onnx.convert",
            "--saved-model",
            export_dir,
            "--output",
            onnx_path,
            "--opset",
            str(opset),
            "--signature_def",
            signature,
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(
                f"tf2onnx.convert failed (exit {proc.returncode}). Command:\n"
                f"{' '.join(cmd)}\n\nstderr:\n{proc.stderr}\nstdout:\n{proc.stdout}"
            )
        return onnx.load(onnx_path)
    finally:
        shutil.rmtree(export_dir, ignore_errors=True)
        try:
            os.remove(onnx_path)
        except OSError:
            pass

### Load data from Feast (offline store) + MinIO

Training features come from **`get_historical_features()`**, which reads Parquet at `s3://<bucket>/feast/credit-fraud/transactions.parquet`.

**Workbench env vars** (any of these naming patterns work; the notebook maps them for Feast):

- `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET`
- or `S3_KEY`, `S3_SECRET`, `S3_ENDPOINT` (your current secret names)

After editing workbench env vars: **Update workbench** → wait for Running → **Restart kernel** in Jupyter.

Prefer in-cluster endpoint: `http://minio-service.shared-s3.svc:9000`

If S3 read fails: `pip install s3fs==2024.9.0`

In [6]:
import os

from feast import FeatureStore

FEAST_REPO = Path("../feast").resolve()
FEAST_CLIENT_CONFIG = Path("/opt/app-root/src/feast-config/credit_fraud")
FEATURE_COLUMNS = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order",
]
TRANSACTION_FEATURES = [f"transaction_features:{col}" for col in FEATURE_COLUMNS]


def _resolve_s3_credentials():
    """Map workbench secret names to standard AWS env vars for pandas + Feast."""
    access_key = (
        os.environ.get("AWS_ACCESS_KEY_ID")
        or os.environ.get("AWS_KEY")
        or os.environ.get("S3_KEY")
    )
    secret_key = (
        os.environ.get("AWS_SECRET_ACCESS_KEY")
        or os.environ.get("AWS_SECRET")
        or os.environ.get("S3_SECRET")
    )
    endpoint = (
        os.environ.get("AWS_S3_ENDPOINT")
        or os.environ.get("AWS_ENDPOINT")
        or os.environ.get("S3_ENDPOINT")
        or "http://minio-service.shared-s3.svc:9000"
    )
    bucket = os.environ.get("AWS_S3_BUCKET", "rhoai")

    if not access_key or not secret_key:
        raise RuntimeError(
            "MinIO credentials not found. Set workbench env vars with names "
            "AWS_ACCESS_KEY_ID + AWS_SECRET_ACCESS_KEY + AWS_S3_ENDPOINT "
            "(or S3_KEY / S3_SECRET / S3_ENDPOINT), then Update workbench and restart the kernel."
        )

    os.environ["AWS_ACCESS_KEY_ID"] = access_key
    os.environ["AWS_SECRET_ACCESS_KEY"] = secret_key
    os.environ["AWS_S3_ENDPOINT"] = endpoint
    os.environ["AWS_ENDPOINT_URL"] = endpoint
    os.environ.setdefault("AWS_S3_BUCKET", bucket)
    return access_key, secret_key, endpoint, bucket


s3_key, s3_secret, s3_endpoint, bucket = _resolve_s3_credentials()
prefix = os.environ.get("FEAST_S3_PREFIX", "feast/credit-fraud")
s3_uri = f"s3://{bucket}/{prefix}/transactions.parquet"

transactions = pd.read_parquet(
    s3_uri,
    storage_options={
        "key": s3_key,
        "secret": s3_secret,
        "client_kwargs": {"endpoint_url": s3_endpoint},
    },
)

entity_df = transactions[["transaction_id", "event_timestamp", "fraud"]].copy()
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"])

if FEAST_CLIENT_CONFIG.is_file():
    store = FeatureStore(fs_yaml_file=str(FEAST_CLIENT_CONFIG))
else:
    store = FeatureStore(repo_path=str(FEAST_REPO))

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=TRANSACTION_FEATURES,
).to_df()

X = training_df[FEATURE_COLUMNS]
y = entity_df["fraud"].to_numpy()

print(f"Loaded {len(X)} rows from Feast (features via S3-backed FileSource)")
X.head()

NoCredentialsError: Unable to locate credentials

### Train / validation / test split and scaling

In [ ]:
# X and y were loaded from Feast above. Split for train/validation/test.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train
)

# Scale the data to remove mean and have unit variance. This means that the data will be between -1 and 1, which makes it a lot easier for the model to learn than random potentially large values.
# It is important to only fit the scaler to the training data, otherwise you are leaking information about the global distribution of variables (which is influenced by the test set) into the training set.

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

# Since the dataset is unbalanced (it has many more non-fraud transactions than fraudulent ones), we set a class weight to weight the few fraudulent transactions higher than the many non-fraud transactions.

class_weights = class_weight.compute_class_weight(
    "balanced", classes=np.unique(y_train), y=y_train
)
class_weights = {i: class_weights[i] for i in range(len(class_weights))}

# y is already a numpy array from the Feast load cell; no .to_numpy() needed.

### Build the DNN (same architecture as the original demo)

In [ ]:
# Build the model, the model we build here is a simple fully connected deep neural network, containing 3 hidden layers and one output layer.

model = Sequential()
model.add(Dense(32, name='dense', activation = 'relu', input_dim = len(X.columns)))
model.add(Dropout(0.2))
model.add(Dense(32, name='dense_02'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(32, name='dense_03'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1, name='dense_04', activation = 'sigmoid'))
model.compile(optimizer='SGD',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

### Configure MLFlow
### Train the model, plot the confusion matrix and push the artifacts to MLFlow.

In [ ]:
# Autolog metrics/params only — do not log or register the TensorFlow/Keras artifact here.
# ONNX is the deployable format for OpenShift AI model serving; we register it manually below.
mlflow.set_experiment("credit-card-fraud")
mlflow.tensorflow.autolog(log_models=False)

### Train, log metrics, export ONNX to MLflow

**Changed from the published demo:** replace `tf2onnx.convert.from_keras(model)` with `keras_model_to_onnx(model)` so ONNX is produced reliably on current OpenShift AI images.

**MLflow layout:** the ONNX bundle is logged under the run artifact **`credit-card-fraud-onnx`** (not the generic name `models`) and registered as **`credit-card-fraud-onnx`**. When you deploy from object storage in OpenShift AI, use the folder path that ends with `artifacts/credit-card-fraud-onnx/`.

In [ ]:
with mlflow.start_run():
    epochs = 2
    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        validation_data=(scaler.transform(X_val), y_val),
        verbose=True,
        class_weight=class_weights,
    )

    y_pred_temp = model.predict(scaler.transform(X_test), verbose=0)
    threshold = 0.995
    y_pred = np.where(y_pred_temp > threshold, 1, 0)
    c_matrix = confusion_matrix(y_test, y_pred)

    ax = sns.heatmap(c_matrix, annot=True, cbar=False, cmap="Blues")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.show()

    t_n, f_p, f_n, t_p = c_matrix.ravel()
    mlflow.log_metric("tn", int(t_n))
    mlflow.log_metric("fp", int(f_p))
    mlflow.log_metric("fn", int(f_n))
    mlflow.log_metric("tp", int(t_p))

    model_proto = keras_model_to_onnx(model)
    # Same label for artifact folder (under the run) and Model Registry entry.
    name = "credit-card-fraud-onnx"
    mlflow.onnx.log_model(
        model_proto,
        artifact_path=name,
        registered_model_name=name,
    )

    scaler_params = {
        "feature_names": FEATURE_COLUMNS,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    }
    scaler_path = FEAST_REPO / "scaler_params.json"
    with open(scaler_path, "w") as scaler_file:
        json.dump(scaler_params, scaler_file, indent=2)
    print(f"Saved scaler params to {scaler_path}")

### Materialize features to the online store (for inference)

After training, copy offline features into the online store so the Gradio app can call `get_online_features()` at inference time. Run in the **feast pod** terminal (`online` container):

```bash
feast materialize 2020-01-01T00:00:00 2026-12-31T00:00:00
```

Or trigger the operator cronjob after pushing updated feature definitions:

```bash
oc create job --from=cronjob/feast-credit-fraud-feast feast-apply-manual-$(date +%s) -n credit-card-fraud
```